# 🚲 Bike Rental Demand Analysis

**Author:** [Your Name]  
**Dataset:** Washington D.C. Bike-Share System (Hourly Records)  
**Tools:** Python · pandas · matplotlib · scikit-learn

---

## Overview

This project explores **what drives bike-sharing demand** across time, weather, and seasonal patterns. Using a dataset of hourly bike rentals from Washington D.C., I investigate:

1. **When do people rent bikes?** How does demand vary by hour and month?
2. **What conditions drive more rentals?** How do temperature, humidity, and weather correlate with demand?
3. **Can demand be predicted?** How well does a linear regression model estimate rental counts?

The goal is to surface actionable patterns that could inform fleet planning, pricing strategy, or station placement for a bike-share operator.

---
## Section 1: Data Loading & Setup

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import sklearn.linear_model

# Global plot style
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False

In [ ]:
df = pd.read_csv('bikes_final.csv')

print(f"Shape: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"\nColumn names: {list(df.columns)}")
df.head()

In [ ]:
# Check for missing values
print("Missing values per column:")
print(df.isnull().sum())
print("\nData types:")
print(df.dtypes)

---
## Section 2: Data Cleaning & Feature Engineering

The `datetime` column is parsed and two new time-based features are extracted:
- **`month`** — captures seasonal patterns (1 = January, 12 = December)
- **`hour`** — captures intraday demand cycles (commute vs. leisure hours)

These engineered features are central to both the visualization and modeling sections.

In [ ]:
df['datetime'] = pd.to_datetime(df['datetime'])
df['month'] = df['datetime'].dt.month
df['hour'] = df['datetime'].dt.hour

print(f"Date range: {df['datetime'].min().date()} to {df['datetime'].max().date()}")
df[['datetime', 'month', 'hour', 'count']].head()

---
## Section 3: Descriptive Statistics

Summary statistics give us a baseline sense of scale. A few things stand out:
- The average rental count per hour is around 190, but the distribution is wide (std ~180), suggesting strong peaks.
- Temperature and humidity are normalized (0–1 scale), not raw Celsius or percentages.

In [ ]:
# Numerical summary
numerical_cols = ['temp', 'humidity', 'windspeed', 'count', 'month', 'hour']
df[numerical_cols].describe().round(2)

In [ ]:
# Categorical summaries
categorical_cols = ['season', 'holiday', 'workingday', 'weather', 'day_of_week']
for col in categorical_cols:
    print(f"--- {col} ---")
    print(df[col].value_counts())
    print()

---
## Section 4: Numerical Distributions

Histograms reveal the shape of each variable's distribution.

**Rental count** is right-skewed — most hours have moderate rentals, but occasional peaks push the tail high. **Temperature** is more evenly spread, reflecting year-round variation across seasons.

In [ ]:
# Reusable helper function for plot formatting
def format_plot(title, xlabel, ylabel, show=True):
    plt.title(title, fontsize=13, fontweight='bold', pad=12)
    plt.xlabel(xlabel, fontsize=11)
    plt.ylabel(ylabel, fontsize=11)
    plt.tight_layout()
    if show:
        plt.show()

# Distribution of bike rentals
df['count'].plot(kind='hist', bins=40, color='steelblue', edgecolor='white', alpha=0.85)
format_plot('Distribution of Hourly Bike Rentals', 'Rentals per Hour', 'Frequency')

In [ ]:
# Distribution of temperature
df['temp'].plot(kind='hist', bins=30, color='coral', edgecolor='white', alpha=0.85)
format_plot('Distribution of Temperature (Normalized)', 'Temperature (0–1 scale)', 'Frequency')

In [ ]:
# Side-by-side boxplots: rentals by season
season_labels = {1: 'Spring', 2: 'Summer', 3: 'Fall', 4: 'Winter'}
df['season_label'] = df['season'].map(season_labels)

season_order = ['Spring', 'Summer', 'Fall', 'Winter']
grouped_data = [df[df['season_label'] == s]['count'].values for s in season_order]

plt.boxplot(grouped_data, labels=season_order, patch_artist=True,
            boxprops=dict(facecolor='steelblue', alpha=0.6))
format_plot('Rental Distribution by Season', 'Season', 'Hourly Rentals')

---
## Section 5: Categorical Data Analysis

Across all four seasons, data is relatively evenly distributed — so seasonal patterns in demand aren't caused by more *observations* in some seasons.

In [ ]:
df['season_label'].value_counts().reindex(season_order).plot(
    kind='bar', color='steelblue', edgecolor='white', alpha=0.85
)
plt.xticks(rotation=0)
format_plot('Record Count by Season', 'Season', 'Number of Records')

In [ ]:
day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']
day_counts = df['day_of_week'].value_counts()

# Reindex if day names are present, otherwise plot as-is
try:
    day_counts = day_counts.reindex(day_order)
except Exception:
    pass

day_counts.plot(kind='bar', color='coral', edgecolor='white', alpha=0.85)
plt.xticks(rotation=30, ha='right')
format_plot('Record Count by Day of Week', 'Day', 'Number of Records')

---
## Section 6: Grouped Analysis

Grouping demand by month reveals a clear **seasonal arc**: rentals rise through spring and peak in summer, then taper off heading into winter. This pattern mirrors typical outdoor recreation and commuting behavior in a temperate climate.

In [ ]:
avg_by_month = df.groupby('month')['count'].mean()

month_names = ['Jan','Feb','Mar','Apr','May','Jun','Jul','Aug','Sep','Oct','Nov','Dec']
avg_by_month.index = month_names

avg_by_month.plot(kind='bar', color='steelblue', edgecolor='white', alpha=0.85)
plt.xticks(rotation=0)
format_plot('Average Hourly Rentals by Month', 'Month', 'Avg Rentals per Hour')

In [ ]:
# Grouped by season and working day
grouped = df.groupby(['season_label', 'workingday'])['count'].mean().unstack()
grouped.columns = ['Weekend/Holiday', 'Working Day']
grouped.reindex(season_order).plot(kind='bar', color=['coral', 'steelblue'],
                                    edgecolor='white', alpha=0.85)
plt.xticks(rotation=0)
plt.legend(frameon=False)
format_plot('Avg Rentals: Working Days vs. Weekends/Holidays by Season',
            'Season', 'Avg Rentals per Hour')

**Observation:** Working-day demand is slightly *higher* than weekend demand — especially in Summer and Fall. This suggests the system is heavily used by commuters, not just recreational riders.

---
## Section 7: Trends Over Time

The hourly demand curve reveals a **bimodal pattern** — two clear peaks around 8 AM and 5–6 PM. This matches typical commute windows and strongly suggests that a large portion of the user base is riding to and from work.

In [ ]:
avg_by_hour = df.groupby('hour')['count'].mean()

plt.plot(avg_by_hour.index, avg_by_hour.values, color='steelblue', linewidth=2.5, marker='o', markersize=4)
plt.axvspan(7, 9, alpha=0.1, color='orange', label='Morning commute')
plt.axvspan(16, 19, alpha=0.1, color='green', label='Evening commute')
plt.legend(frameon=False)
plt.xticks(range(0, 24))
format_plot('Average Hourly Bike Rentals Throughout the Day', 'Hour of Day', 'Avg Rentals')

In [ ]:
# Compare hourly patterns: working day vs. weekend
avg_by_hour_wd = df[df['workingday'] == 1].groupby('hour')['count'].mean()
avg_by_hour_we = df[df['workingday'] == 0].groupby('hour')['count'].mean()

plt.plot(avg_by_hour_wd.index, avg_by_hour_wd.values, label='Working Day', color='steelblue', linewidth=2)
plt.plot(avg_by_hour_we.index, avg_by_hour_we.values, label='Weekend/Holiday', color='coral', linewidth=2)
plt.xticks(range(0, 24))
plt.legend(frameon=False)
format_plot('Hourly Demand: Working Days vs. Weekends', 'Hour of Day', 'Avg Rentals')

**Key insight:** On weekdays, demand spikes sharply at commute hours. On weekends, the pattern flattens into a single midday peak — consistent with leisurely daytime riding.

---
## Section 8: Variable Relationships

Before modeling, I examine pairwise correlations and scatter plots to identify which features relate most strongly to rental count.

In [ ]:
numerical_cols = ['temp', 'humidity', 'windspeed', 'count', 'month', 'hour']
corr = df[numerical_cols].corr()

print("Correlation with 'count':")
print(corr['count'].sort_values(ascending=False).round(3))

In [ ]:
plt.scatter(df['temp'], df['count'], alpha=0.15, s=8, color='steelblue')
format_plot('Temperature vs. Bike Rentals', 'Temperature (Normalized)', 'Hourly Rentals')

In [ ]:
plt.scatter(df['humidity'], df['count'], alpha=0.15, s=8, color='coral')
format_plot('Humidity vs. Bike Rentals', 'Humidity (Normalized)', 'Hourly Rentals')

**Observations:**
- Temperature has the strongest individual correlation with rentals — a positive but curved relationship (rentals drop slightly at extreme heat).
- Humidity shows a mild negative relationship: higher humidity tends to correspond with fewer rentals.
- No single variable dominates — demand is genuinely multifactorial.

---
## Section 9: Hypothesis

> **Hypothesis:** Bike rental demand is jointly driven by temperature, time of day, and day type (working day vs. weekend). Higher temperatures and peak commute hours will increase demand, while high humidity will suppress it. Workdays will show sharper intraday peaks than weekends due to commuter behavior.

This hypothesis motivates the multivariable regression model in the next section.

---
## Section 10: Linear Regression

Two regression models are built:

1. **Simple regression** — temperature only. Used to visualize the regression line.
2. **Multivariable regression** — temperature, humidity, windspeed, month, and hour as predictors.

In [ ]:
# ── Simple regression: temperature → count ──
X_simple = df[['temp']]
y = df['count']

model_simple = sklearn.linear_model.LinearRegression()
model_simple.fit(X_simple, y)
y_pred_simple = model_simple.predict(X_simple)

sorted_idx = df['temp'].argsort()

plt.scatter(df['temp'], df['count'], alpha=0.15, s=6, color='steelblue', label='Actual')
plt.plot(df['temp'].iloc[sorted_idx], y_pred_simple[sorted_idx],
         color='red', linewidth=2, label='Regression line')
plt.legend(frameon=False)
format_plot('Simple Regression: Temperature → Bike Rentals',
            'Temperature (Normalized)', 'Hourly Rentals')

print(f"Coefficient: {model_simple.coef_[0]:.2f}  |  Intercept: {model_simple.intercept_:.2f}")
print(f"R² (simple): {model_simple.score(X_simple, y):.3f}")

In [ ]:
# ── Multivariable regression ──
feature_cols = ['temp', 'humidity', 'windspeed', 'month', 'hour']
X_multi = df[feature_cols]

model_multi = sklearn.linear_model.LinearRegression()
model_multi.fit(X_multi, y)
y_pred_multi = model_multi.predict(X_multi)

plt.scatter(y, y_pred_multi, alpha=0.15, s=6, color='steelblue')
plt.plot([y.min(), y.max()], [y.min(), y.max()], 'r--', linewidth=1.5, label='Perfect fit')
plt.legend(frameon=False)
format_plot('Multivariable Regression: Actual vs. Predicted Rentals',
            'Actual Rentals', 'Predicted Rentals')

print(f"R² (multivariable): {model_multi.score(X_multi, y):.3f}")
print("\nFeature coefficients:")
for feat, coef in zip(feature_cols, model_multi.coef_):
    print(f"  {feat:<12} {coef:+.2f}")

---
## Section 11: Regression Discussion

**What the results suggest:**
- The simple temperature model (R² ≈ 0.16) captures only a modest share of variance — temperature alone is a weak predictor.
- Adding month, hour, humidity, and windspeed improves the fit meaningfully, confirming that demand is driven by a combination of time and weather factors.
- The coefficient signs align with intuition: higher temperature → more rentals; higher humidity → fewer rentals; later hours (up to evening peak) → more rentals.

**Limitations:**
- Linear regression assumes straight-line relationships, but temperature and hour both have curved effects on demand.
- Categorical variables (weather code, season, day of week) were excluded from the model — encoding them could improve R².
- The model is trained and evaluated on the same data, so the R² is optimistic (no train/test split was applied).

**Better approaches:** A gradient-boosted tree or random forest model could capture non-linear interactions and likely achieve R² > 0.8 on held-out data, as shown in published work on this dataset.

---
## Section 12: Conclusion

This analysis surfaced three clear patterns in Washington D.C. bike-share data:

1. **Demand is time-driven.** The bimodal hourly curve (8 AM and 5–6 PM peaks) reveals heavy commuter use. Weekends show a different, flatter midday pattern — recreational in nature.

2. **Weather matters — especially temperature.** Rentals rise with temperature up to a point, while high humidity suppresses demand. Wind speed has a weaker effect.

3. **Linear regression captures the direction but not the full story.** The multivariable model explains a meaningful share of variance, but the non-linear nature of many relationships limits its accuracy. More powerful models are the logical next step.

**If I were advising the bike-share operator:** stock more bikes at stations near offices for morning and evening commutes, and build inventory plans around the June–August demand peak. Weather-based dynamic pricing could also be explored — demand is measurably elastic to temperature and weather conditions.